# Lesson 5 — Orchestrator (Multi‑Agent) with LangGraph Router → A2A Calls → Merged Final Answer

**Audience:** University faculty (Malaysia)  
**Goal:** Build an **Orchestrator** that routes a single user query across **2–3 A2A agents**, then merges outputs into one final response.

We will use **LangGraph** because it makes routing + multi-call workflows explicit and teachable.

---

## What you will build

### Agents (already built earlier)
- **CoursePolicyAgent** (Lesson 1) — `http://127.0.0.1:9999/`
- **CampusInfoToolAgent** (Lesson 3) — `http://127.0.0.1:9998/`
- **WebResearchAgent** (Lesson 4) — `http://127.0.0.1:9997/`

### New in this lesson
1. A **router** that decides which agents to call  
2. A **multi-call plan** (e.g., call 2 agents for one question)  
3. A **merger** that formats the final output in a structured way:
   - **Final Answer**
   - **Evidence**
   - **Actions / Next steps**
   - **Trace** (inputs/outputs of each agent)

---

## Prerequisites (start these servers in terminals)

Terminal 1:
```bash
python a2a_course_policy_agent.py
```

Terminal 2:
```bash
python a2a_campus_info_agent.py
```

Terminal 3:
```bash
python a2a_web_research_agent.py
```

> Notebook tip: use top-level `await` (not `asyncio.run`).


## 1. Setup

Install (if needed):
- `langgraph`
- `a2a-sdk` client libs
- `httpx`


In [ ]:
# If needed, uncomment and run:
!pip -q install "langgraph>=0.2.0" a2a-sdk httpx python-dotenv

In [1]:
import os
from dotenv import load_dotenv

_ = load_dotenv()
print("✅ Environment ready")

✅ Environment ready


## 2. A2A client helper (call any agent + capture trace)

We will:
- Discover AgentCard
- Send a message
- Return:
  - `agent_name`
  - `prompt`
  - `response_text`
  - `card` (basic fields)

This trace will be displayed in Markdown so faculty can see **exactly** what happened.


In [24]:
import httpx
from typing import Dict, Any

from a2a.client import ClientConfig, ClientFactory, create_text_message_object
from a2a.types import AgentCard, Artifact, Message, Task
from a2a.utils.message import get_message_text

async def call_a2a_agent(prompt: str, base_url: str) -> Dict[str, Any]:
    async with httpx.AsyncClient(timeout=180.0) as httpx_client:
        client = await ClientFactory.connect(
            base_url,
            client_config=ClientConfig(httpx_client=httpx_client),
        )
        card: AgentCard = await client.get_card()

        message = create_text_message_object(content=prompt)
        responses = client.send_message(message)

        text_content = ""
        async for response in responses:
            if isinstance(response, Message):
                text_content = get_message_text(response)
            elif isinstance(response, tuple):
                task: Task = response[0]
                if task.artifacts:
                    artifact: Artifact = task.artifacts[0]
                    text_content = get_message_text(artifact)

        return {
            "agent_name": card.name,
            "agent_description": card.description,
            "agent_url": card.url,
            "skills": [s.name for s in (card.skills or [])],
            "prompt": prompt,
            "response_text": text_content,
        }

## 3. Pretty trace display (Markdown)

We want professors to clearly see:
- what the orchestrator decided
- which agents were called
- each agent’s input/output
- the final merged result


In [25]:
from IPython.display import Markdown, display

def md_escape(s: str) -> str:
    return (s or "").replace("```", "``\u200b`")

def render_agent_trace(trace: Dict[str, Any]) -> None:
    md = []
    md.append(f"## 🔎 Agent Call Trace — **{trace['agent_name']}**")
    md.append(f"**Description:** {trace['agent_description']}")
    md.append(f"**URL:** `{trace['agent_url']}`")
    if trace.get("skills"):
        md.append("**Skills:** " + ", ".join(trace["skills"]))
    md.append("")
    md.append("### Input")
    md.append(f"```text\n{md_escape(trace['prompt'])}\n```")
    md.append("")
    md.append("### Output")
    md.append(f"```text\n{md_escape(trace['response_text'])}\n```")
    display(Markdown("\n".join(md)))

def render_orchestrator_plan(plan: Dict[str, Any]) -> None:
    r = plan["route"]
    md = []
    md.append("# 🧭 Orchestrator Plan (LLM Router)")
    md.append(f"**User question:** `{plan['question']}`\n")
    md.append("### Router decision")
    md.append(f"- **Policy needed:** {r['policy']}")
    md.append(f"- **Campus info needed:** {r['campus']}")
    md.append(f"- **Web research needed:** {r['research']}")
    md.append(f"\n**Why:** {r.get('why','')}\n")
    md.append("### Calls to execute")
    for c in plan["calls"]:
        md.append(f"- **{c['agent']}** — prompt: `{c['prompt']}`")
    display(Markdown("\n".join(md)))

def render_final_output(final_md: str) -> None:
    display(Markdown(final_md))

## 4. Define the routing policy (simple, teachable)

We will route using a **small LLM router** (optional) or simple heuristics.

For professors, heuristics are easiest to understand first.

### Routing rules (v1)
- If question mentions **policy/rubric/late/extension/plagiarism/marks** → call CoursePolicyAgent
- If question mentions **staff/office hours/room/timetable/contact** → call CampusInfoToolAgent
- If question mentions **recent/latest/guidelines/best practices** OR if policy agent returns “Not found” → call WebResearchAgent


In [30]:
import re
from typing import List

from openai import OpenAI
import json
from typing import Any, Dict, Optional
from openai import OpenAI

POLICY_URL = "http://127.0.0.1:9999"
CAMPUS_URL = "http://127.0.0.1:9998"
RESEARCH_URL = "http://127.0.0.1:9997"




router_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Use a strong model for routing (you can override via env)
ROUTER_MODEL = os.environ.get("ROUTER_MODEL", "gpt-5")  # or "gpt-5-mini" if you prefer cheaper

ROUTER_SYSTEM = """
You are a router/planner for a university multi-agent assistant.

You have 3 agents:
1) CoursePolicyAgent: answers ONLY from the course policy handbook PDF.
2) CampusInfoToolAgent: answers ONLY from campus MCP tools (rooms, labs, staff, office hours, timetable, contacts).
3) WebResearchAgent: uses live web search and returns citations.

Your task:
- Decide which agents should be called for the user question.
- Produce agent-specific sub-questions (prompts) so each agent receives ONLY what it can answer.
- If the question is about a lab/room like "CS-Lab-2" or "CS-3.21" -> campus agent.
- If the question is about penalties, submissions, rubrics, marks, extensions -> policy agent.
- Use web research ONLY if the answer is not expected in policy PDF or campus tools, or as fallback.

Return ONLY valid JSON with keys exactly:
{
  "policy": boolean,
  "campus": boolean,
  "research": boolean,
  "policy_prompt": string|null,
  "campus_prompt": string|null,
  "research_prompt": string|null,
  "why": string
}

Rules:
- If policy is false, policy_prompt must be null.
- If campus is false, campus_prompt must be null.
- If research is false, research_prompt must be null.
- Keep prompts short and focused (one intent each).
"""

def llm_route_and_decompose(question: str) -> Dict[str, Any]:
    resp = router_client.responses.create(
        model=ROUTER_MODEL,
        input=[
            {"role": "system", "content": ROUTER_SYSTEM},
            {"role": "user", "content": f"USER QUESTION:\n{question}\n\nReturn JSON only."},
        ],
    )

    raw = (resp.output_text or "").strip()
    try:
        obj = json.loads(raw)
    except Exception:
        # Fallback if router ever returns non-JSON
        obj = {
            "policy": False,
            "campus": False,
            "research": True,
            "policy_prompt": None,
            "campus_prompt": None,
            "research_prompt": question,
            "why": f"Router JSON parse failed. Raw: {raw[:200]}",
        }

    # Hard validation / normalization
    def b(x): return bool(x)
    obj["policy"] = b(obj.get("policy"))
    obj["campus"] = b(obj.get("campus"))
    obj["research"] = b(obj.get("research"))

    if not obj["policy"]:
        obj["policy_prompt"] = None
    if not obj["campus"]:
        obj["campus_prompt"] = None
    if not obj["research"]:
        obj["research_prompt"] = None

    # If it didn't select anything, default to research
    if not (obj["policy"] or obj["campus"] or obj["research"]):
        obj["research"] = True
        obj["research_prompt"] = question
        obj["why"] = (obj.get("why") or "") + " | Defaulted to research."

    return obj

def build_plan(question: str) -> Dict[str, Any]:
    route = llm_route_and_decompose(question)

    calls = []
    if route["policy"]:
        calls.append({"agent": "CoursePolicyAgent", "url": POLICY_URL, "prompt": route["policy_prompt"]})
    if route["campus"]:
        calls.append({"agent": "CampusInfoToolAgent", "url": CAMPUS_URL, "prompt": route["campus_prompt"]})
    if route["research"]:
        calls.append({"agent": "WebResearchAgent", "url": RESEARCH_URL, "prompt": route["research_prompt"]})

    return {"question": question, "route": route, "calls": calls}

#### HELPER FUNCTION

In [31]:
import re

def extract_section(text: str, header: str) -> str:
    """
    Extract a section like:
      Answer:
      ...
      Evidence:
      ...
    Returns the text between 'header:' and the next top-level header.
    """
    if not text:
        return ""
    pattern = rf"{header}\s*:\s*(.*?)(?=\n[A-Z][A-Za-z /]*\s*:|\Z)"
    m = re.search(pattern, text, flags=re.S | re.M)
    return m.group(1).strip() if m else ""

def normalize_bullets(block: str) -> str:
    """
    Keep bullets nicely, but if the block is plain text, wrap as a bullet.
    """
    if not block:
        return ""
    lines = [ln.rstrip() for ln in block.splitlines() if ln.strip()]
    # If already has bullets, keep as-is
    if any(ln.lstrip().startswith(("-", "•")) for ln in lines):
        return "\n".join(lines)
    # otherwise make it a bullet
    return "- " + " ".join(lines)

## 5. Build the LangGraph Orchestrator

We implement a simple graph:

1) **Planner**: creates a plan (which agents to call)  
2) **Caller**: executes each A2A call and stores traces  
3) **Fallback**: if policy response contains `Not found in the provided document.` → call WebResearchAgent  
4) **Merger**: produce final structured response + include trace blocks

This demonstrates *multi-agent orchestration* cleanly.


In [32]:
from langgraph.graph import StateGraph, END

# State keys:
# - question: str
# - plan: dict
# - traces: list[dict]
# - final: str

def planner_node(state):
    question = state["question"]
    plan = build_plan(question)
    return {"question": question, "plan": plan, "traces": []}

async def caller_node(state):
    question = state["question"]
    plan = state["plan"]
    traces = state.get("traces", [])

    for c in plan["calls"]:
        t = await call_a2a_agent(c["prompt"], c["url"])
        traces.append(t)

    return {"question": question, "plan": plan, "traces": traces}

async def fallback_node(state):
    question = state["question"]
    plan = state["plan"]
    traces = state.get("traces", [])

    # Only fallback if a *called agent* failed on its own sub-question
    need_research = False

    for t in traces:
        out = (t.get("response_text") or "")
        if t["agent_name"] == "CoursePolicyAgent" and out.strip() == "Not found in the provided document.":
            need_research = True
        if t["agent_name"] == "CampusInfoToolAgent" and ("error" in out.lower() or "cannot find" in out.lower()):
            need_research = True

    # Run research only if needed AND not already called by router
    already_called = any(t["agent_name"] == "WebResearchAgent" for t in traces)
    if need_research and not already_called:
        traces.append(await call_a2a_agent(question, RESEARCH_URL))

    return {"question": question, "plan": plan, "traces": traces}


def merger_node(state):
    question = state["question"]
    traces = state.get("traces", [])

    policy = next((t for t in traces if t["agent_name"] == "CoursePolicyAgent"), None)
    campus = next((t for t in traces if t["agent_name"] == "CampusInfoToolAgent"), None)
    research = next((t for t in traces if t["agent_name"] == "WebResearchAgent"), None)

    policy_answer_block = normalize_bullets(extract_section(policy["response_text"], "Answer")) if policy else ""
    campus_answer_block = normalize_bullets(extract_section(campus["response_text"], "Answer")) if campus else ""

    final = []
    final.append("# ✅ Final Answer")
    final.append(f"**User question:** `{question}`\n")

    final.append("## Answer")

    if policy_answer_block:
        final.append("### Late penalty (policy)")
        final.append(policy_answer_block)

    if campus_answer_block:
        final.append("\n### Office hours (campus tools)")
        final.append(campus_answer_block)

    # --- Evidence ---
    final.append("\n## Evidence")
    if policy:
        final.append("- From CoursePolicyAgent (course handbook).")
    if campus:
        final.append("- From CampusInfoToolAgent (MCP tools).")
    if research:
        final.append("- From WebResearchAgent (web sources with citations).")

    # --- Actions ---
    final.append("\n## Actions / Next steps")
    final.append("- If you need confirmation, verify the policy page reference and the staff office-hour record.")

    # --- Optional trace (still keep for teaching) ---
    final.append("\n## Trace (agent inputs/outputs)")
    for t in traces:
        final.append(f"\n### {t['agent_name']}")
        final.append("**Input**")
        final.append(f"```text\n{md_escape(t['prompt'])}\n```")
        final.append("**Output**")
        final.append(f"```text\n{md_escape(t['response_text'])}\n```")

    return {"question": question, "final": "\n".join(final)}

orch = StateGraph(dict)
orch.add_node("planner", planner_node)
orch.add_node("caller", caller_node)
orch.add_node("fallback", fallback_node)
orch.add_node("merger", merger_node)

orch.set_entry_point("planner")
orch.add_edge("planner", "caller")
orch.add_edge("caller", "fallback")
orch.add_edge("fallback", "merger")
orch.add_edge("merger", END)

orchestrator = orch.compile()
print("✅ LangGraph Orchestrator compiled")

✅ LangGraph Orchestrator compiled


## 6. Run the orchestrator

We will:
1) Render the **plan**
2) Execute multi-agent calls
3) Render each agent trace (optional)
4) Render the final merged output


In [33]:
question = "What is the late penalty after 2 days, and what are Dr Amina Rahman's office hours?"
state = {"question": question}

# Show plan before execution
plan = build_plan(question)
render_orchestrator_plan(plan)

out = await orchestrator.ainvoke({"question": question})
render_final_output(out["final"])



# 🧭 Orchestrator Plan (LLM Router)
**User question:** `What is the late penalty after 2 days, and what are Dr Amina Rahman's office hours?`

### Router decision
- **Policy needed:** True
- **Campus info needed:** True
- **Web research needed:** False

**Why:** Late penalties come from the course policy handbook, while instructor office hours are in campus tools. No web search is required.

### Calls to execute
- **CoursePolicyAgent** — prompt: `What is the late submission penalty after 2 days?`
- **CampusInfoToolAgent** — prompt: `What are the current office hours for Dr Amina Rahman?`

# ✅ Final Answer
**User question:** `What is the late penalty after 2 days, and what are Dr Amina Rahman's office hours?`

## Answer
### Late penalty (policy)
- The late submission penalty for work submitted 2 days late is a 20% deduction, resulting in a maximum of 80% of the total marks available.

### Office hours (campus tools)
- Dr. Amina Rahman's current office hours are as follows:
  - Tuesday: 14:00-16:00 (In-person) at CS-3.21
  - Thursday: 10:00-11:00 (Online) via MS Teams

## Evidence
- From CoursePolicyAgent (course handbook).
- From CampusInfoToolAgent (MCP tools).

## Actions / Next steps
- If you need confirmation, verify the policy page reference and the staff office-hour record.

## Trace (agent inputs/outputs)

### CoursePolicyAgent
**Input**
```text
According to the course policy handbook PDF, what is the late submission penalty for work submitted 2 days late?
```
**Output**
```text
Answer: The late submission penalty for work submitted 2 days late is a 20% deduction, resulting in a maximum of 80% of the total marks available.

Evidence:
- "2 days (25–48 hrs) 20% deducted 80% of total marks" (Page 4)
- "Includes weekends" (Page 4)
```

### CampusInfoToolAgent
**Input**
```text
Using campus tools, what are Dr Amina Rahman's current office hours (days/times and location)?
```
**Output**
```text
Answer:
- Dr. Amina Rahman's current office hours are as follows:
  - Tuesday: 14:00-16:00 (In-person) at CS-3.21
  - Thursday: 10:00-11:00 (Online) via MS Teams

Evidence:
- Tool: get_office_hours
- Key fields: staff_name, office_hours

Actions / Next steps:
- If you need further assistance or information, please let me know!
```

## 7. Try more prompts (faculty-friendly)

### A) Only campus info
- “Where is CS-Lab-2 and what facilities does it have?”

### B) Only policy
- “What is the process to appeal a marking decision?”

### C) Policy not found → fallback to research
- “What are common university policies for generative AI disclosure in assignments?”
  - (If your course handbook does not mention it explicitly, the orchestrator should automatically call WebResearchAgent.)


In [34]:
# Try your own:
question = "Where is CS-Lab-2 and what facilities does it have?"
render_orchestrator_plan(build_plan(question))
out = await orchestrator.ainvoke({"question": question})
render_final_output(out["final"])

# 🧭 Orchestrator Plan (LLM Router)
**User question:** `Where is CS-Lab-2 and what facilities does it have?`

### Router decision
- **Policy needed:** False
- **Campus info needed:** True
- **Web research needed:** False

**Why:** The question is about the location and amenities of a specific lab (CS-Lab-2), which are provided by campus information tools, not the course policy or external web sources.

### Calls to execute
- **CampusInfoToolAgent** — prompt: `Provide the building/room location for CS-Lab-2 and list its facilities/equipment, capacity, opening hours, and any booking or access requirements.`

# ✅ Final Answer
**User question:** `Where is CS-Lab-2 and what facilities does it have?`

## Answer

### Office hours (campus tools)
- The CS-Lab-2 is located in the CS building. It has a capacity of 35 and is equipped with PCs, a whiteboard, and HDMI connectivity.

## Evidence
- From CampusInfoToolAgent (MCP tools).

## Actions / Next steps
- If you need confirmation, verify the policy page reference and the staff office-hour record.

## Trace (agent inputs/outputs)

### CampusInfoToolAgent
**Input**
```text
Get CS-Lab-2 room details: exact location (building/floor/room), map link, and facilities (workstations/specs, OS/software, capacity, peripherals/printers, projector/boards, accessibility, and access hours).
```
**Output**
```text
Answer:
- The CS-Lab-2 is located in the CS building. It has a capacity of 35 and is equipped with PCs, a whiteboard, and HDMI connectivity.

Evidence:
- Tool: find_room
- Key fields: building, room, capacity, facilities

Actions / Next steps:
- For more detailed information such as the exact floor, map link, OS/software, peripherals/printers, projector/boards, accessibility, and access hours, please check the university's room booking system or contact the facilities management office.
```

In [35]:
question = "What is the process to appeal a marking decision?"
render_orchestrator_plan(build_plan(question))
out = await orchestrator.ainvoke({"question": question})
render_final_output(out["final"])

# 🧭 Orchestrator Plan (LLM Router)
**User question:** `What is the process to appeal a marking decision?`

### Router decision
- **Policy needed:** True
- **Campus info needed:** False
- **Web research needed:** False

**Why:** Appeals about marking/grades are governed by the course policy handbook (assessment appeals), not campus logistics or external web info.

### Calls to execute
- **CoursePolicyAgent** — prompt: `From the course policy handbook, what is the process to appeal a marking decision? Include eligibility, steps, deadlines/time limits, required documentation, who to contact/submit to, and escalation if unresolved.`

# ✅ Final Answer
**User question:** `What is the process to appeal a marking decision?`

## Answer
### Late penalty (policy)
- The process to appeal a marking decision involves three steps: an informal review, a formal re-mark request, and an appeal to the Student Appeals Committee. Students must first contact the marker or Unit Coordinator within 5 business days. If unresolved, they can submit a formal re-mark request within 10 business days, with a fee of AUD 50. If still unsatisfied, they can appeal to the Student Appeals Committee within 20 business days of the re-mark outcome.

## Evidence
- From CoursePolicyAgent (course handbook).

## Actions / Next steps
- If you need confirmation, verify the policy page reference and the staff office-hour record.

## Trace (agent inputs/outputs)

### CoursePolicyAgent
**Input**
```text
From the course policy handbook, what is the process to appeal a marking decision (steps, deadlines, who to contact, required form/evidence, and escalation path)?
```
**Output**
```text
Answer: The process to appeal a marking decision involves three steps: an informal review, a formal re-mark request, and an appeal to the Student Appeals Committee. Students must first contact the marker or Unit Coordinator within 5 business days. If unresolved, they can submit a formal re-mark request within 10 business days, with a fee of AUD 50. If still unsatisfied, they can appeal to the Student Appeals Committee within 20 business days of the re-mark outcome.

Evidence:
- "Contact the marker or Unit Coordinator within 5 business days of receiving the result." (Page 14)
- "Submit a Formal Re-mark Request through the LMS within 10 business days of the original result." (Page 14)
- "Appeals must be lodged within 20 business days of the formal re-mark outcome." (Page 14)
- "Email p.nair@northgate.edu.au with your student ID, the assessment item, and a clear explanation of your concern." (Page 14)
```

In [36]:
question = "What are common university policies for generative AI disclosure in assignments?"
render_orchestrator_plan(build_plan(question))
out = await orchestrator.ainvoke({"question": question})
render_final_output(out["final"])

# 🧭 Orchestrator Plan (LLM Router)
**User question:** `What are common university policies for generative AI disclosure in assignments?`

### Router decision
- **Policy needed:** False
- **Campus info needed:** False
- **Web research needed:** True

**Why:** The user asks about general, cross-university norms for AI disclosure, which are not specific to a single course policy or campus tool. This requires web research across multiple universities.

### Calls to execute
- **WebResearchAgent** — prompt: `Find common university policies on generative AI use in assignments with emphasis on disclosure. Summarize typical requirements (allowed vs. prohibited use, citation/attribution, documenting prompts/edits, instructor permission, consequences). Provide 5–7 example policies from different universities with citations.`

# ✅ Final Answer
**User question:** `What are common university policies for generative AI disclosure in assignments?`

## Answer

## Evidence
- From WebResearchAgent (web sources with citations).

## Actions / Next steps
- If you need confirmation, verify the policy page reference and the staff office-hour record.

## Trace (agent inputs/outputs)

### WebResearchAgent
**Input**
```text
Survey recent university policies on generative AI use in assignments and summarize common disclosure requirements. Include: allowed vs. restricted use, required disclosure elements (tool/model/version, prompts, extent of use), citation/attribution wording, data/privacy cautions, integrity/honor code notes, and consequences. Provide 5–7 representative citations from major universities from the last 2–3 years.
```
**Output**
```text
Recent university policies on the use of generative AI in assignments exhibit a range of approaches, from strict prohibitions to encouraged integration, each emphasizing transparency, ethical use, and academic integrity. Below is a summary of common disclosure requirements and practices:

**Allowed vs. Restricted Use:**

- **Permitted Use:** Some institutions allow the use of generative AI tools for specific purposes, such as brainstorming, drafting, or enhancing content, provided students adhere to guidelines set by instructors. For example, Cornell University permits AI use for identified assignments, with clear directions indicating when and how AI can be utilized. ([teaching.cornell.edu](https://teaching.cornell.edu//generative-artificial-intelligence/ai-academic-integrity?utm_source=openai))

- **Prohibited Use:** Other universities, like Elon University, explicitly prohibit the use of AI generators unless otherwise noted by the instructor, emphasizing that all submitted work must be original and completed without AI assistance. ([elon.edu](https://www.elon.edu/u/academics/writing-excellence/2024/08/06/ai-writing-assignment-syllabus-policies/?utm_source=openai))

**Required Disclosure Elements:**

- **Tool/Model/Version:** Students are often required to specify the AI tool used. For instance, the University of North Carolina at Charlotte mandates that students acknowledge the use of a generative AI tool in their assignments, detailing how it was used (e.g., brainstorming, grammatical correction). ([legal.charlotte.edu](https://legal.charlotte.edu/faqs/policies-use-generative-artificial-intelligence-ai?utm_source=openai))

- **Prompts:** Some policies require students to disclose the prompts used when interacting with AI tools. This practice ensures transparency in how AI-generated content was obtained.

- **Extent of Use:** Universities like the University of Texas at Dallas expect students to document and attribute the use of generative AI, specifying the extent and manner of its application in their academic work. ([policy.utdallas.edu](https://policy.utdallas.edu/utdsp5017?utm_source=openai))

**Citation/Attribution Wording:**

- Proper citation is emphasized across institutions. For example, Cornell University advises students to attribute directly quoted text to the creator of the generative AI tool used, citing OpenAI when quoting ChatGPT. ([teaching.cornell.edu](https://teaching.cornell.edu//generative-artificial-intelligence/ai-academic-integrity?utm_source=openai))

**Data/Privacy Cautions:**

- Universities caution against sharing sensitive information with AI tools, as data provided may be used for training AI models or other purposes, potentially compromising confidentiality. The University of North Carolina at Charlotte highlights the importance of avoiding sharing personally identifiable information, protected health information, financial data, and other sensitive data when using AI tools. ([legal.charlotte.edu](https://legal.charlotte.edu/faqs/policies-use-generative-artificial-intelligence-ai?utm_source=openai))

**Integrity/Honor Code Notes:**

- Adherence to academic integrity is a common theme. Institutions like the University of Texas at Dallas state that misuse of generative AI that results in academic dishonesty will be subject to disciplinary action as outlined in the Student Code of Conduct and other applicable university rules. ([policy.utdallas.edu](https://policy.utdallas.edu/utdsp5017?utm_source=openai))

**Consequences:**

- Consequences for misuse vary but can include penalties ranging from assignment point deductions to failure of the course, depending on the severity of the infraction. For example, the University of North Alabama treats misuse of AI as a violation of academic integrity, potentially resulting in penalties from assignment point deductions to course failure. ([una.edu](https://www.una.edu/academics/generative-ai-policy.html?utm_source=openai))

**Representative Citations from Major Universities:**

1. **University of North Alabama**: "Generative AI Policy" (2023). ([una.edu](https://www.una.edu/academics/generative-ai-policy.html?utm_source=openai))

2. **Middle Tennessee State University**: "323 Instructional and Assignment Use of Artificial Intelligence – University Policies" (2024). ([mtsu.edu](https://www.mtsu.edu/policies/323-instructional-and-assignment-use-of-artificial-intelligence/?utm_source=openai))

3. **Cornell University**: "AI & Academic Integrity | Center for Teaching Innovation" (2023). ([teaching.cornell.edu](https://teaching.cornell.edu//generative-artificial-intelligence/ai-academic-integrity?utm_source=openai))

4. **University of Texas at Dallas**: "UTDSP5017 Generative AI Use in Academic Work | Policy Navigator" (2023). ([policy.utdallas.edu](https://policy.utdallas.edu/utdsp5017?utm_source=openai))

5. **Elon University**: "Developing AI Writing Assignments & Syllabus Policies | Center for Writing Excellence" (2023). ([elon.edu](https://www.elon.edu/u/academics/writing-excellence/2024/08/06/ai-writing-assignment-syllabus-policies/?utm_source=openai))

6. **University of North Carolina at Charlotte**: "Policies on the use of generative artificial intelligence (AI) | Office of Legal Affairs" (2023). ([legal.charlotte.edu](https://legal.charlotte.edu/faqs/policies-use-generative-artificial-intelligence-ai?utm_source=openai))

7. **University of Maine**: "Generative AI Teaching and Learning Guidelines - Community Standards, Rights, and Responsibilities" (2023). ([umaine.edu](https://umaine.edu/communitystandards/resources-for-faculty/generative-ai-teaching-and-learning-guidelines/?utm_source=openai))

These policies reflect a growing recognition of the role of generative AI in education, emphasizing the need for clear guidelines to ensure ethical and responsible use.

Sources:
- Source — https://www.una.edu/academics/generative-ai-policy.html?utm_source=openai
- Source — https://www.mtsu.edu/policies/323-instructional-and-assignment-use-of-artificial-intelligence/?utm_source=openai
- Source — https://teaching.cornell.edu//generative-artificial-intelligence/ai-academic-integrity?utm_source=openai
- Source — https://sce.cornell.edu/courses/students/policies/use-of-ai?utm_source=openai
- Source — https://www.uona.edu/ai/policy/faculty-focus?utm_source=openai
- Source — https://www.uona.edu/ai/policy/student-guide?utm_source=openai

Notes:
- (Auto-added sources from tool metadata.)
```

## 8. Exercises (for faculty participants)

1) Improve routing:
- Add keywords for “consultation hour” → campus tool agent

2) Add “multi-step” plan:
- If question contains BOTH policy + campus, call both (already done).
- Add: if policy returns Not found, call research (already done).

3) Add a “confidence note”:
- In the final output, include which agent was most authoritative:
  - Policy for policy/rubric rules
  - Campus tools for staff/rooms/timetable
  - Web research for general best practices

4) Stretch: LLM-based router
- Replace heuristics with a small LLM router that returns JSON: {policy: bool, campus: bool, research: bool}
